# QMedViT Unified Notebook

Single end-to-end pipeline for classical and quantum MedViT variants. Pick a variant in the config cell below; the rest of the notebook is identical to the per-size classical notebooks (install → dataset → train → test → FGSM/PGD adversarial eval).

**Available `MODEL_VARIANT` options:**

| Variant | Train | Test | Description |
| --- | --- | --- | --- |
| `classical_small` | — | — | Baseline `MedViT.MedViT_small` |
| `classical_base` | — | — | Baseline `MedViT.MedViT_base` |
| `classical_large` | — | — | Baseline `MedViT.MedViT_large` |
| `quantum_softmax_gpu_gpu` | analytic sim | analytic sim | Trains and evaluates with the fast exact-statevector path (`qpu_mode=False`). Differentiable through the quantum softmax. |
| `quantum_softmax_gpu_qpu` | analytic sim | shots sim (`qpu_shots=5000`) | Trains analytically, then rebuilds the model in `qpu_mode=True` and copies weights for eval — mimics deploying a trained model on a noisy/shot-limited backend. |
| `quantum_softmax_qpu_qpu` | shots sim (`qpu_shots=5000`) | shots sim (`qpu_shots=5000`) | Trains and evaluates with finite-shot sampling. Quantum softmax weights are frozen during training (QPU path detaches gradients); the rest of the network still trains. Very slow. |
| `quantum_quanv_stem` | classical SGD | classical SGD | Replaces the first ConvBNReLU of the stem (3→64) with a Quanvolutional layer (Henderson et al. 2019, arXiv:1904.04767). 64 random 9-qubit circuits sampled once and frozen; outputs precomputed into a 512-entry lookup table per filter. The quantum part is non-trainable — only the BN/ReLU after the quanv layer and all classical layers downstream are trained. |
| `quantum_quanv_stem_gpu_qpu` | analytic table (init) | shots-sampled table (init, `qpu_shots=5000`) | Lookup table built via analytic statevector at training-model construction, but rebuilt with finite-shot sampling at eval-model construction. The forward path stays a lookup in both cases — QPU mode adds shot noise to the table VALUES, not the forward call. Random circuits are seeded identically so train/eval share the same projection up to sampling noise. |

The "GPU" path is PennyLane's `default.qubit` analytic statevector simulator (no shots — the model is differentiable); the "QPU" path is the same device with `shots=5000`, mimicking what real hardware does (sampled probabilities, no gradients through the quantum block).

## Install Requirements

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/alexandrachirita98/MedViT-Quantum/

In [ ]:
%cd /kaggle/working/MedViT-Quantum

In [ ]:
%pwd

In [ ]:
pip install -r requirements.txt

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data

import torchvision
import torchvision.utils
from torchvision import models
import torchvision.datasets as dsets
import torchvision.transforms as transforms
from torchsummary import summary

from tqdm import tqdm
import medmnist
from medmnist import INFO, Evaluator

import torchattacks
from torchattacks import PGD, FGSM

In [ ]:
print("PyTorch", torch.__version__)
print("Torchvision", torchvision.__version__)
print("Torchattacks", torchattacks.__version__)
print("Numpy", np.__version__)
print("Medmnist", medmnist.__version__)

## Configuration

Edit this cell to choose the model variant, dataset, and training hyperparameters. Everything below this cell is variant-agnostic.

In [ ]:
MODEL_VARIANT = "quantum_quanv_stem"

DATA_FLAG = "retinamnist"
# [tissuemnist, pathmnist, chestmnist, dermamnist, octmnist,
# pnemoniamnist, retinamnist, breastmnist, bloodmnist, tissuemnist,
# organamnist, organcmnist, organsmnist]

NUM_EPOCHS = 10
BATCH_SIZE = 10
LR = 0.005

QPU_SHOTS = 5000

from MedViT import MedViT_small, MedViT_base, MedViT_large
from quantum_variants.softmax_only import QMedViT_Softmax_Only
from quantum_variants.quanv_stem import QMedViT_Quanv_Stem


def _qsoftmax(qpu, n):
    return QMedViT_Softmax_Only(
        stem_chs=[64, 32, 64], depths=[3, 4, 10, 3], path_dropout=0.1,
        num_classes=n, qpu_mode=qpu, qpu_shots=QPU_SHOTS,
    )


def _qquanv_stem(n, seed=0):
    return QMedViT_Quanv_Stem(
        stem_chs=[64, 32, 64], depths=[3, 4, 10, 3], path_dropout=0.1,
        num_classes=n, quanv_seed=seed,
    )


def _qquanv_stem_qpu(n, seed=0):
    return QMedViT_Quanv_Stem(
        stem_chs=[64, 32, 64], depths=[3, 4, 10, 3], path_dropout=0.1,
        num_classes=n, quanv_seed=seed,
        qpu_mode=True, qpu_shots=QPU_SHOTS,
    )


MODEL_BUILDERS = {
    "classical_small": {"train": lambda n: MedViT_small(num_classes=n), "eval_swap": None},
    "classical_base":  {"train": lambda n: MedViT_base(num_classes=n),  "eval_swap": None},
    "classical_large": {"train": lambda n: MedViT_large(num_classes=n), "eval_swap": None},
    "quantum_softmax_gpu_gpu": {
        "train": lambda n: _qsoftmax(False, n),
        "eval_swap": None,
    },
    "quantum_softmax_gpu_qpu": {
        "train": lambda n: _qsoftmax(False, n),
        "eval_swap": lambda n: _qsoftmax(True, n),
    },
    "quantum_softmax_qpu_qpu": {
        "train": lambda n: _qsoftmax(True, n),
        "eval_swap": None,
    },
    "quantum_quanv_stem": {
        "train": lambda n: _qquanv_stem(n),
        "eval_swap": None,
    },
    "quantum_quanv_stem_gpu_qpu": {
        "train": lambda n: _qquanv_stem(n),
        "eval_swap": lambda n: _qquanv_stem_qpu(n),
    },
}

if MODEL_VARIANT not in MODEL_BUILDERS:
    raise ValueError(
        f"Unknown MODEL_VARIANT '{MODEL_VARIANT}'. "
        f"Expected one of: {list(MODEL_BUILDERS)}"
    )

print(f"Selected variant: {MODEL_VARIANT}")

## Dataset

In [ ]:
data_flag = DATA_FLAG
download = True

info = INFO[data_flag]
task = info['task']
n_channels = info['n_channels']
n_classes = len(info['label'])

DataClass = getattr(medmnist, info['python_class'])

print("number of channels : ", n_channels)
print("number of classes : ", n_classes)

In [ ]:
from torchvision.transforms.transforms import Resize
# preprocessing
train_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.Lambda(lambda image: image.convert('RGB')),
    torchvision.transforms.AugMix(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])
test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.Lambda(lambda image: image.convert('RGB')),
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])

# load the data
train_dataset = DataClass(split='train', transform=train_transform, download=download)
test_dataset = DataClass(split='test', transform=test_transform, download=download)

# encapsulate data into dataloader form
train_loader = data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
train_loader_at_eval = data.DataLoader(dataset=train_dataset, batch_size=2*BATCH_SIZE, shuffle=False)
test_loader = data.DataLoader(dataset=test_dataset, batch_size=2*BATCH_SIZE, shuffle=False)

In [ ]:
print(train_dataset)
print("===================")
print(test_dataset)

## Model

In [ ]:
model = MODEL_BUILDERS[MODEL_VARIANT]["train"](n_classes).cuda()

## Train

In [ ]:
# define loss function and optimizer
if task == "multi-label, binary-class":
    criterion = nn.BCEWithLogitsLoss()
else:
    criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(model.parameters(), lr=LR, momentum=0.9)

In [ ]:
# train

for epoch in range(NUM_EPOCHS):
    train_correct = 0
    train_total = 0
    test_correct = 0
    test_total = 0
    print('Epoch [%d/%d]'% (epoch+1, NUM_EPOCHS))
    model.train()
    for inputs, targets in tqdm(train_loader):
        inputs, targets = inputs.cuda(), targets.cuda()
        # forward + backward + optimize
        optimizer.zero_grad()
        outputs = model(inputs)

        if task == 'multi-label, binary-class':
            targets = targets.to(torch.float32)
            loss = criterion(outputs, targets)
        else:
            targets = targets.squeeze().long()
            loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

## Test

In [ ]:
# If the selected variant changes execution mode between train and test
# (e.g. quantum_softmax_gpu_qpu: trained on the analytic simulator, evaluated
# with finite shots), rebuild the model in eval mode and copy weights over.
_eval_swap = MODEL_BUILDERS[MODEL_VARIANT]["eval_swap"]
if _eval_swap is not None:
    _new_model = _eval_swap(n_classes).cuda()
    _new_model.load_state_dict(model.state_dict())
    model = _new_model
    print(f"Swapped model to eval mode for variant: {MODEL_VARIANT}")
else:
    print(f"No eval-mode swap needed for variant: {MODEL_VARIANT}")

In [ ]:
split = 'test'

model.eval()
y_true = torch.tensor([])
y_score = torch.tensor([])

data_loader = train_loader_at_eval if split == 'train' else test_loader

with torch.no_grad():
    for inputs, targets in data_loader:
        inputs = inputs.cuda()
        outputs = model(inputs)
        outputs = outputs.softmax(dim=-1)
        y_score = torch.cat((y_score, outputs.cpu()), 0)

    y_score = y_score.detach().numpy()

    evaluator = Evaluator(data_flag, split, size=224)
    metrics = evaluator.evaluate(y_score)

    print('%s  auc: %.3f  acc: %.3f' % (split, *metrics))

## Adversarial Robustness

reduce batch size for GPU limitation

In [ ]:
BATCH_SIZE = 5
test_loader = data.DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
model.eval()

correct = 0
total = 0

atk = FGSM(model, eps=0.01)

for images, labels in test_loader:
    labels = labels.squeeze(1)
    images = atk(images, labels).cuda()
    outputs = model(images)

    _, predicted = torch.max(outputs.data, 1)

    total += labels.size(0)
    correct += (predicted == labels.cuda()).sum()

print('FGSM Robust accuracy: %.2f %%' % (100 * float(correct) / total))

In [ ]:
model.eval()

correct = 0
total = 0

atk = PGD(model, eps=8/255, alpha=4/255, steps=10, random_start=True)

for images, labels in test_loader:
    labels = labels.squeeze(1)
    images = atk(images, labels).cuda()
    outputs = model(images)

    _, predicted = torch.max(outputs.data, 1)

    total += labels.size(0)
    correct += (predicted == labels.cuda()).sum()

print('PGD Robust accuracy: %.2f %%' % (100 * float(correct) / total))